# Imports

In [39]:
import torch
import torch.nn as nn

from torchvision import datasets
from torchvision import transforms

# Dataset

In [40]:
transform = transforms.ToTensor()

dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform
)

# Get Two Images

In [41]:
image1, _ = dataset[0]
image2, _ = dataset[1]

# Add batch dimension: 

image1 = image1.unsqueeze(0)
image2 = image2.unsqueeze(0)

print(image1.shape)

torch.Size([1, 3, 32, 32])


# Encoder

In [42]:
class Encoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(
                in_channels=3,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(
                32 * 8 * 8,
                128
            )
        )

    def forward(self, x):
        return self.encoder(x)

# Predictor

In [43]:
class Predictor(nn.Module):

    def __init__(self):
        super().__init__()

        self.predictor = nn.Sequential(

            nn.Linear(128, 256),

            nn.ReLU(),

            nn.Linear(256, 128)
        )

    def forward(self, x):
        return self.predictor(x)

# Create Models

In [44]:
online_encoder = Encoder()

target_encoder = Encoder()

predictor = Predictor()

# Copy Weights

In [45]:
target_encoder.load_state_dict(
    online_encoder.state_dict()
)

<All keys matched successfully>

# Freeze Target Encoder

In [46]:
for param in target_encoder.parameters():
    param.requires_grad = False

# Loss Function

In [47]:
loss_fn = nn.MSELoss()

# Optimizer

In [48]:
optimizer = torch.optim.Adam(
    list(online_encoder.parameters()) +
    list(predictor.parameters()),
    lr=0.001
)

# Forward Pass

### Online side

In [49]:
z1 = online_encoder(image1)

pred_z2 = predictor(z1)

### Target side

In [50]:
with torch.no_grad():

    z2 = target_encoder(image2)

In [51]:
print(z1.shape)
print(pred_z2.shape)
print(z2.shape)

torch.Size([1, 128])
torch.Size([1, 128])
torch.Size([1, 128])


# Compute Loss

In [52]:
loss = loss_fn(
    pred_z2,
    z2
)

print(loss)

tensor(0.0067, grad_fn=<MseLossBackward0>)


# Backpropagation

In [53]:
optimizer.zero_grad()
loss.backward()
optimizer.step()

Online Encoder  ✓ updated

Predictor       ✓ updated

Target Encoder  ✗ unchanged

# EMA Update

In [54]:
m = 0.99

for target_param, online_param in zip(
    target_encoder.parameters(),
    online_encoder.parameters()
):

    target_param.data = (
        m * target_param.data
        +
        (1 - m) * online_param.data
    )

# Full Architecture

image 1 -> online Encoder -> z1 -> predictor -> pred_z2

image 2 -> target Encoder -> z2

MSE loss

# Training Loop

In [55]:
for epoch in range(100):
    
    # find pred z2 using z1
    z1 = online_encoder(image1)
    pred_z2 = predictor(z1)

    # find actual z2
    with torch.no_grad():
        z2 = target_encoder(image2)
    
    # find loss
    loss = loss_fn(z2, pred_z2)

    # This update online_encoder
    optimizer.zero_grad() # Clear the old math.
    loss.backward() # This function performs the actual backpropagation, but it only does the math.
    optimizer.step() # This is where the actual updating happens.

    # update target_encoder
    for target_param, online_param in zip(
        target_encoder.parameters(),
        online_encoder.parameters()
    ):
        
        # formula to update target_encoder a small step
        # like .2 .2 .2
        target_param.data = (m * target_param) + ((1-m) * online_param)

    print(epoch, loss.item())



0 0.004010674078017473
1 0.0024519332218915224
2 0.0014835742767900229
3 0.0008127051987685263
4 0.00046800501877442
5 0.0003265502746216953
6 0.0002656914875842631
7 0.00022668200836051255
8 0.0002198903530370444
9 0.00022445397917181253
10 0.00023811295977793634
11 0.00024171348195523024
12 0.00024626741651445627
13 0.0002431764150969684
14 0.0002378836798015982
15 0.00022819353034719825
16 0.00021284456306602806
17 0.0001959412475116551
18 0.0001715897669782862
19 0.00014169882342685014
20 0.0001145045825978741
21 8.874060586094856e-05
22 6.50465372018516e-05
23 4.773317414219491e-05
24 3.724173075170256e-05
25 3.190770803485066e-05
26 2.870920070563443e-05
27 2.5352688680868596e-05
28 2.2208472728380002e-05
29 2.0013967514387332e-05
30 1.9176368368789554e-05
31 1.9429171516094357e-05
32 2.177162423322443e-05
33 3.0672119464725256e-05
34 3.674947583931498e-05
35 2.6786838134285063e-05
36 1.1315047231619246e-05
37 2.3966065782587975e-05
38 2.6213710953015834e-05
39 8.367897862626705e